# Optimising individual prices with `open-rating-engine`

This notebook starts from the pipeline saved by `02_optimization_rating_factor.ipynb` and sets **one price multiplier per vehicle** for next year's renewals, to maximise expected profit while holding the conversion rate at the level the current rates would give.

Individual prices are only set for the renewal cohort. New business is priced on the rating factors alone, so this notebook works only with the in-force book, aged forward to its next renewal.

| Step | What it does |
|---|---|
| Age the book | Rolls this year's in-force book forward a year with `apply_aging`, giving next year's renewal cohort |
| Baseline | Loads the rating-factor-optimised pipeline and scores the renewal cohort |
| Simulate | Re-scores the book across a grid of price multipliers, recording conversion and profit for every vehicle at every price point |
| Optimise | `PriceOptimizerIndividual` picks one grid multiplier per vehicle, so that the book's average conversion hits the target |
| Efficient frontier | Repeats the optimisation across a range of conversion targets, to show the profit/volume trade-off |
| Deploy | Writes the chosen multipliers into the pipeline's per-vehicle rating table, re-scores the book and saves the pipeline |

Rating factors move whole groups of vehicles together. The individual optimiser has no such restriction: each vehicle gets the price point that best balances its own profit against its own chance of converting. It works on a fixed grid, so every vehicle's multiplier is one of the grid factors, and it never calls the pipeline itself.

**Prerequisites**

- Run `01_rating_algorithm_example.ipynb` and `02_optimization_rating_factor.ipynb` first. Notebook 02 saves the baseline pipeline used here to `models/rating_engines/pricing_pipeline_rating_factor_optimized.pkl`.
- Install the package with the ONNX, H2O and notebook extras. From the `open-rating-engine` folder: `pip install -e ".[onnx,h2o,notebook]"`
- Java 8+ installed, with `JAVA_HOME` set. The demand model is an H2O MOJO, scored in a JVM.
- Run the notebook from the `.examples` folder, so the `data/` and `models/` folders resolve.

In [ ]:
from copy import deepcopy
from pathlib import Path

import cloudpickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mlflow.pyfunc import PythonModelContext

from open_rating_engine.data_processing import apply_aging
from open_rating_engine.optimization import PriceOptimizerIndividual, make_log_multiplier_penalty
from open_rating_engine.rating_engine import RatingTablesLayer, SourceColumn

# Imported from the submodule: once it's loaded, the package attribute of the same
# name resolves to the module rather than the function.
from open_rating_engine.optimization.simulate_adjustment_grid import (
    make_demand_emulation,
    simulate_adjustment_grid,
)

## 1. Input data

The in-force book: this year's policies, both renewals (`RN`) and those written as new business (`NB`). `risk.gwp` is the premium each vehicle is paying this year.

In [ ]:
DATA_DIR = Path("data").resolve() / "renewal"

in_force = {
    table: pd.read_parquet(DATA_DIR / f"{table}.parquet")
    for table in ["policy", "risk", "cover", "claims_history"]
}

pd.DataFrame({"rows": {table: len(df) for table, df in in_force.items()}})

In [ ]:
in_force["policy"]["policy.transaction_type"].value_counts()

## 2. Age the book to next year's renewal

Every in-force policy comes up for renewal next year, so the cohort to price is this book moved forward one year. The aging rules say how each source column moves:

| Column | Next year |
|---|---|
| `risk.vehicle_age`, `risk.driver_youngest_age`, `policy.customer_tenure` | + 1 |
| `policy.effective_date`, `policy.expiry_date` | + 1 year |
| `policy.transaction_type` | `"RN"` for every policy |
| `risk.previous_gwp` | this year's premium, `risk.gwp` |
| `risk.vehicle_market_value` | × 0.95 |

`apply_aging` takes the rules as `SourceColumn` layer configs, each with an `aging_rule` holding a pandas expression. It evaluates each expression with `df` set to the column's table, and writes the result back to that column. It ages the frames in place, so it gets a copy of the in-force book.

The origination cohort has its own rules: it only rolls the policy dates forward a year and stamps `NB`, leaving the risk attributes at their raw values. It isn't priced individually, so it isn't used here.

In [ ]:
def aging_rule(table, column, expression, dtype):
    config = SourceColumn(name=column, source_table=table, source_column=column, dtype=dtype).to_dict()
    config["aging_rule"] = {"expression": expression, "enabled": True}
    return config


RENEWAL_AGING_RULES = {
    rule["name"]: rule
    for rule in [
        aging_rule("risk", "risk.vehicle_age", "df['risk.vehicle_age'] + 1", "numeric"),
        aging_rule("risk", "risk.driver_youngest_age", "df['risk.driver_youngest_age'] + 1", "numeric"),
        aging_rule("policy", "policy.customer_tenure", "df['policy.customer_tenure'] + 1", "numeric"),
        aging_rule(
            "policy", "policy.effective_date",
            "pd.to_datetime(df['policy.effective_date']) + pd.DateOffset(years=1)", "date",
        ),
        aging_rule(
            "policy", "policy.expiry_date",
            "pd.to_datetime(df['policy.expiry_date']) + pd.DateOffset(years=1)", "date",
        ),
        aging_rule("policy", "policy.transaction_type", '"RN"', "categorical"),
        aging_rule("risk", "risk.previous_gwp", "df['risk.gwp']", "numeric"),
        aging_rule("risk", "risk.vehicle_market_value", "df['risk.vehicle_market_value'] * 0.95", "numeric"),
    ]
}

data = apply_aging(RENEWAL_AGING_RULES, {table: df.copy() for table, df in in_force.items()})

The first policy and its first vehicle, before and after aging:

In [ ]:
pd.DataFrame({
    column: {
        "In force": in_force[rule["source_table"]][column].iloc[0],
        "Next year's renewal": data[rule["source_table"]][column].iloc[0],
    }
    for column, rule in RENEWAL_AGING_RULES.items()
}).T

In [ ]:
data["policy"]["policy.transaction_type"].value_counts()

## 3. Baseline rating engine

The pipeline was saved with `cloudpickle`, so it holds its layers, catalog and data structure, but not its loaded models. The ONNX and H2O layers reload their model files in `load_context`.

As in notebook 02, a fresh context built from this folder replaces the saved one, whose artifact paths point at the machine that saved the pipeline.

In [ ]:
MODELS_DIR = Path("models").resolve()
BASELINE_PATH = MODELS_DIR / "rating_engines" / "pricing_pipeline_rating_factor_optimized.pkl"

with BASELINE_PATH.open("rb") as file:
    pricing_engine = cloudpickle.load(file)

artifacts = {
    "motor_risk_cost_onnx": MODELS_DIR / "motor_risk_cost_onnx" / "pipeline_onnx.onnx",
    "motor_demand_mojo": MODELS_DIR / "motor_demand_h2o" / "model.zip",
    "h2o_gen_model": MODELS_DIR / "motor_demand_h2o" / "h2o-genmodel.jar",
}
missing = [key for key, path in artifacts.items() if not path.exists()]
assert not missing, f"Model files not found for {missing}. Run this notebook from open-rating-engine/.examples."

context = PythonModelContext(
    artifacts={key: str(path) for key, path in artifacts.items()},
    model_config={"environment": "local", "java_options": "-Xmx1g"},
)
pricing_engine.load_context(context)

# The grid below scores the book many times over; one progress bar per score is noise.
pricing_engine.options["show_progress"] = False

print(f"{len(pricing_engine.get_layer_names())} layers loaded")

### The per-vehicle multiplier

The demand premium is the technical premium × `risk.demand_multiplier` (the rating factors tuned in notebook 02) × `risk.demand_multiplier_individual`. The individual layer is a `RatingTablesLayer` with a single table keyed on `risk.risk_number`. In the baseline it holds one placeholder row that matches no vehicle, so every vehicle takes the default of 1.0.

In [ ]:
LAYER_NAME = "risk.demand_multiplier_individual"

individual_layer = pricing_engine.layers[LAYER_NAME]
print(f"formula: {individual_layer.formula}")
print(f"default_values: {individual_layer.default_values}")
individual_layer.rating_tables["risk_number"]

In [ ]:
print(pricing_engine.layers["risk.premium_demand_total"].calculation)

### Score the baseline

The baseline is next year's renewal cohort priced at the current rates, with every individual multiplier at 1.0.

In [ ]:
baseline_output = pricing_engine.predict(data)

with pd.option_context("display.max_columns", None):
    display(baseline_output["risk"].head(10))

Book-level KPIs, as averages per vehicle. *Expected* figures are weighted by the probability that the customer accepts the quote, so expected profit is the profit the book makes once the quotes that don't convert are taken out.

In [ ]:
KPI_COLUMNS = {
    "Offered GWP": "risk.gwp",
    "Conversion rate": "risk.conversion_rate",
    "Expected GWP": "risk.reporting_expected_gwp",
    "Expected profit": "risk.reporting_expected_profit",
}


def book_kpis(scored):
    risk = scored["risk"]
    kpis = {label: risk[column].mean() for label, column in KPI_COLUMNS.items()}
    kpis["Expected profit margin"] = kpis["Expected profit"] / kpis["Expected GWP"]
    return pd.Series(kpis)


baseline_kpis = book_kpis(baseline_output)
baseline_kpis.to_frame("Baseline").round(4)

## 4. Simulate the price response

`simulate_adjustment_grid` scores the book once for each adjustment factor and returns an `(N, K)` array for every requested column: one row per vehicle, one column per factor.

`make_demand_emulation` in `"override"` mode sets every value in the `risk_number` table, and its default, to the factor. Every vehicle falls back to the default, so each column of the grid is the book priced with every individual multiplier set to that factor.

**Choosing the grid.** The optimiser can only choose multipliers that are on the grid, so the grid sets both the range of price moves allowed and their granularity. Here it runs from 0.90 to 1.10 in steps of 0.01. Each factor is a full score of the book on a deep copy of the pipeline, run in parallel threads. This step takes a few minutes.

In [ ]:
OUTPUT_COLUMNS = {
    "gwp": "risk.gwp",
    "conversion_rate": "risk.conversion_rate",
    "expected_gwp": "risk.reporting_expected_gwp",
    "expected_profit": "risk.reporting_expected_profit",
}

premium_emulation = make_demand_emulation(
    layer_name=LAYER_NAME,
    table_key="risk_number",
    mode="override",
)

simulated = simulate_adjustment_grid(
    data=data,
    pipeline=pricing_engine,
    premium_emulation=premium_emulation,
    output_table="risk",
    output_columns=OUTPUT_COLUMNS,
    adjustment_factor=np.round(np.arange(0.90, 1.1001, 0.01), 2),
    parallel_n_jobs=5,
)

factors = simulated["adjustment_factor_array"]
print(f"factors: {factors}")
print(f"conversion_rate array: {simulated['conversion_rate'].shape} (vehicles x factors)")

Averaging the grid over the book gives the response to a **uniform** price change: every vehicle's multiplier set to the same factor. The 1.00 column reproduces the baseline KPIs.

In [ ]:
uniform_response = pd.DataFrame(
    {name: simulated[name].mean(axis=0) for name in OUTPUT_COLUMNS},
    index=pd.Index(factors, name="adjustment_factor"),
)


def at_factor(factor):
    # Book-average KPIs when every vehicle's multiplier is `factor`.
    return uniform_response.loc[np.isclose(uniform_response.index, factor)].iloc[0]


baseline_conversion = at_factor(1.0)["conversion_rate"]
print(f"Baseline conversion: {baseline_conversion:.2%}")

uniform_response.round(4)

### Vehicles held by the cap and floor

After aging, every vehicle has a previous premium: this year's. The renewal premium can't rise above the cap or fall below the floor relative to it (the `assumptions_cap_cup` catalog table, by tenure band). When the demand premium is already outside those limits, the offered premium stays at the limit at every factor on the grid, and so do the vehicle's conversion and profit. The optimiser can't change anything for these vehicles.

In [ ]:
gwp_grid = simulated["gwp"]
held = pd.Series(np.isclose(gwp_grid[:, 0], gwp_grid[:, -1]), name="held")
tenure_band = baseline_output["risk"]["risk.customer_tenure_band"].reset_index(drop=True)

cap_cup = pricing_engine.catalog["assumptions_cap_cup"].set_index("customer_tenure_band")
held_by_band = held.groupby(tenure_band, observed=True).agg(vehicles="size", share_held="mean")
cap_cup.join(held_by_band).round(4)

## 5. Optimise

### Objective and penalty

`PriceOptimizerIndividual` takes the arrays directly: an objective to maximise and a constraint to hold at a target, both `(N, K)`. For each vehicle it picks the grid column that maximises `objective − λ × constraint`, and finds λ by bisection so that the book's average constraint equals the target.

The objective is each vehicle's expected profit less a penalty on the size of its move. `make_log_multiplier_penalty` returns `l1 × |log k| + l2 × (log k)²` for each grid factor `k`, in the objective's units (dollars of expected profit per vehicle). It treats a 5% increase and the matching decrease alike.

The penalty matters here beyond keeping moves small. A vehicle held by its cap or floor has the same objective at every factor, and the optimiser breaks ties by taking the first column, the 0.90 discount. Any positive penalty sends those vehicles to 1.00 instead. With `l2 = 100`, a 10% move costs about $0.91 of expected profit per vehicle, and a 1% move about a cent.

In [ ]:
penalty = make_log_multiplier_penalty(l1_penalty=0.0, l2_penalty=100.0)(factors)  # (K,)

pd.Series(penalty, index=pd.Index(factors, name="adjustment_factor"), name="penalty").round(4).to_frame().T

In [ ]:
optimizer = PriceOptimizerIndividual(
    adjustment_factors=factors,
    objective_array=simulated["expected_profit"] - penalty,  # broadcasts (K,) across (N, K)
    constraint_array=simulated["conversion_rate"],
    reporting_arrays={
        "gwp": simulated["gwp"],
        "expected_gwp": simulated["expected_gwp"],
        "expected_profit": simulated["expected_profit"],
    },
)

### Maximise profit at the baseline conversion rate

In the summary, `objective` is the average penalised objective and `expected_profit` is the average expected profit without the penalty. `price_multiplier` is the average chosen multiplier.

`lagrangian_lambda` is the interval the bisection searches for λ, the expected profit given up per unit of conversion. The default of ±1,000 is too narrow for the top of the efficient frontier in section 6. The floor stops most premiums from falling far, so near the top a price cut buys little conversion, and the highest target needs λ of about −2,300.

In [ ]:
LAMBDA_RANGE = (-1e5, 1e5)

result = optimizer.optimize(
    target_constraint=float(baseline_conversion),
    tol=1e-4,
    lagrangian_lambda=LAMBDA_RANGE,
)

result["summary"].T

### Where the optimiser moved price

The largest group stays at 1.00: every vehicle held by the cap or floor, plus those whose gain from a move doesn't cover the penalty. The rest spread across the grid, with a pile-up at the 1.10 edge from vehicles that would take a larger increase if the grid allowed one.

In [ ]:
optimal_multipliers = result["optimal_multipliers"]

multiplier_counts = pd.Series(optimal_multipliers).value_counts().reindex(factors, fill_value=0)

BAR_COLOR = "#2a78d6"
TEXT_COLOR = "#52514e"

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(multiplier_counts.index, multiplier_counts.to_numpy(), width=0.008, color=BAR_COLOR)
ax.set_xticks(factors[::2])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:,.0f}"))
ax.set_xlabel("Optimal multiplier", color=TEXT_COLOR)
ax.set_ylabel("Vehicles", color=TEXT_COLOR)
ax.set_title("Optimal multiplier per vehicle", loc="left")
ax.grid(axis="y", color="#e4e3df", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.tick_params(colors=TEXT_COLOR)

plt.tight_layout()
plt.show()

In [ ]:
moves = pd.DataFrame({
    "customer_tenure_band": baseline_output["risk"]["risk.customer_tenure_band"].to_numpy(),
    "multiplier": optimal_multipliers,
})
moves.groupby("customer_tenure_band", observed=True).agg(
    vehicles=("multiplier", "size"),
    mean_multiplier=("multiplier", "mean"),
    share_decreased=("multiplier", lambda m: (m < 1).mean()),
    share_unchanged=("multiplier", lambda m: (m == 1).mean()),
    share_increased=("multiplier", lambda m: (m > 1).mean()),
).reindex(cap_cup.index).round(4)

## 6. Efficient frontier

Re-running the optimisation across a range of conversion targets traces the most expected profit available at each level of conversion. The range runs from the conversion of a uniform 9% increase to that of a uniform 9% decrease. The ends of the grid are left out: a uniform 0.90 gives the most conversion the book can reach, so a target there has a single solution, which the bisection can miss.

In [ ]:
target_range = np.linspace(at_factor(1.09)["conversion_rate"], at_factor(0.91)["conversion_rate"], 9)

frontier = optimizer.calculate_efficient_frontier(
    target_constraint_range=target_range,
    tol=1e-4,
    lagrangian_lambda=LAMBDA_RANGE,
)

frontier.round(4)

The chart compares the frontier with the uniform price change from section 4. Setting every vehicle to the same factor is one of the solutions open to the optimiser, so the frontier sits on or above the uniform line. The penalty can pull it slightly below the line at the extremes, where the uniform change moves every vehicle a long way.

In [ ]:
UNIFORM_COLOR = "#eb6834"
FRONTIER_COLOR = "#2a78d6"

optimized = result["summary"].iloc[0]

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    uniform_response["conversion_rate"],
    uniform_response["expected_profit"],
    color=UNIFORM_COLOR, linewidth=2, marker="o", markersize=5,
    label="Uniform price change",
)
ax.plot(
    frontier["constraint"],
    frontier["expected_profit"],
    color=FRONTIER_COLOR, linewidth=2, marker="o", markersize=5,
    label="Individual prices (frontier)",
)
ax.scatter(
    [baseline_conversion], [baseline_kpis["Expected profit"]],
    s=90, color="white", edgecolor=TEXT_COLOR, linewidth=2, zorder=3,
)
ax.annotate(
    "Baseline", (baseline_conversion, baseline_kpis["Expected profit"]),
    xytext=(-14, -18), textcoords="offset points", ha="right", color=TEXT_COLOR,
)
ax.scatter(
    [optimized["constraint"]], [optimized["expected_profit"]],
    s=90, color=FRONTIER_COLOR, edgecolor="white", linewidth=2, zorder=4,
)
ax.annotate(
    "Optimised", (optimized["constraint"], optimized["expected_profit"]),
    xytext=(-12, -14), textcoords="offset points", ha="right", color=TEXT_COLOR,
)

ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"-${-y:,.0f}" if y < 0 else f"${y:,.0f}"))
ax.set_xlabel("Conversion rate", color=TEXT_COLOR)
ax.set_ylabel("Expected profit per vehicle", color=TEXT_COLOR)
ax.set_title("Expected profit vs conversion", loc="left")
ax.grid(color="#e4e3df", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.tick_params(colors=TEXT_COLOR)
ax.legend(frameon=False, loc="upper right")

plt.tight_layout()
plt.show()

## 7. Deploy the individual prices

The chosen multipliers become the `risk_number` table of a new `RatingTablesLayer`, one row per vehicle. It keeps the original layer's name, formula, default, dependencies and output column, so a vehicle that isn't in the table still takes 1.0. The grid arrays are in the row order of the scored `risk` table, which is where the keys come from.

Swapping the layer into a copy of the pipeline leaves the baseline untouched for comparison.

In [ ]:
risk_keys = baseline_output["risk"]["risk.risk_number"].to_numpy()
assert len(risk_keys) == len(optimal_multipliers)

per_vehicle_table = pd.DataFrame({
    "risk.risk_number": risk_keys,
    "value": optimal_multipliers,
})

optimized_layer = RatingTablesLayer(
    name=individual_layer.name,
    target_table=individual_layer.target_table,
    output_column=individual_layer.output_column,
    rating_tables={"risk_number": per_vehicle_table},
    formula=individual_layer.formula,
    default_values=individual_layer.default_values,
    dependencies=individual_layer.dependencies,
    description=individual_layer.description,
)

optimized_engine = deepcopy(pricing_engine)
optimized_engine.remove_parameter(LAYER_NAME)
optimized_engine.add_parameter(optimized_layer)

optimized_engine.layers[LAYER_NAME].rating_tables["risk_number"].head(10)

### Re-score the book

Scoring with the real pipeline checks the optimiser's estimate. They should agree to rounding: every chosen multiplier is a grid point, so there's no interpolation, and each vehicle's conversion and profit depend only on its own premium.

In [ ]:
optimized_output = optimized_engine.predict(data)

with pd.option_context("display.max_columns", None):
    display(optimized_output["risk"].head(10))

In [ ]:
optimized_kpis = book_kpis(optimized_output)

comparison = pd.DataFrame({
    "Baseline": baseline_kpis,
    "Optimised (re-scored)": optimized_kpis,
    "Optimiser estimate": {
        "Offered GWP": optimized["gwp"],
        "Conversion rate": optimized["constraint"],
        "Expected GWP": optimized["expected_gwp"],
        "Expected profit": optimized["expected_profit"],
        "Expected profit margin": optimized["expected_profit"] / optimized["expected_gwp"],
    },
})
comparison["Change vs baseline"] = comparison["Optimised (re-scored)"] / comparison["Baseline"] - 1
comparison.round(4)

In [ ]:
print(f"Conversion:      {baseline_kpis['Conversion rate']:.2%} -> {optimized_kpis['Conversion rate']:.2%}")
print(f"Expected profit: ${baseline_kpis['Expected profit']:.2f} -> ${optimized_kpis['Expected profit']:.2f} per vehicle "
      f"({optimized_kpis['Expected profit'] / baseline_kpis['Expected profit'] - 1:+.2%})")
print(f"Profit margin:   {baseline_kpis['Expected profit margin']:.2%} -> {optimized_kpis['Expected profit margin']:.2%}")

### Results by tenure band

Tenure band sets the cap and floor, so it also sets how far the optimiser could move each vehicle's price.

In [ ]:
def kpis_by(scored, column):
    risk = scored["risk"]
    return risk.groupby(column, observed=True).agg(
        vehicles=("risk.risk_number", "count"),
        individual_multiplier=("risk.demand_multiplier_individual", "mean"),
        conversion_rate=("risk.conversion_rate", "mean"),
        expected_profit=("risk.reporting_expected_profit", "mean"),
    )


pd.concat(
    {
        "Baseline": kpis_by(baseline_output, "risk.customer_tenure_band"),
        "Optimised": kpis_by(optimized_output, "risk.customer_tenure_band"),
    },
    axis=1,
).reindex(cap_cup.index).round(4)

## 8. Save the optimised pipeline

Saved with `cloudpickle` beside the baseline, in the same way as notebooks 01 and 02. Load it with `cloudpickle.load` and call `load_context` before scoring.

The per-vehicle table is specific to this book. Scoring other vehicles with this pipeline gives them the default multiplier of 1.0.

In [ ]:
OPTIMIZED_PATH = MODELS_DIR / "rating_engines" / "pricing_pipeline_individual_optimized.pkl"

with OPTIMIZED_PATH.open("wb") as file:
    cloudpickle.dump(optimized_engine, file)

print(f"Pipeline saved to {OPTIMIZED_PATH}")